# Residual Direction Tracking

How do directions in the residual stream evolve through layers?
Direction persistence, change rate, unembedding trajectory, and
dominant direction evolution.

## Why This Matters

Residual stream direction tracking characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_direction_tracking import (
    direction_persistence, direction_change_rate,
    unembed_direction_trajectory, dominant_direction_evolution,
    residual_direction_tracking_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Direction Persistence

How much does each layer's direction persist to later layers?

In [ ]:
result = direction_persistence(model, tokens, position=-1)
for p in result['per_layer']:
    bar = '█' * int((p['mean_persistence'] + 1) * 20)
    print(f"  Layer {p['layer']}: mean={p['mean_persistence']:.4f}, "
          f"min={p['min_persistence']:.4f} {bar}")

## Direction Change Rate

Angle of change between consecutive layers.

In [ ]:
result = direction_change_rate(model, tokens, position=-1)
print(f"Mean angle: {result['mean_angle']:.1f}°, max: {result['max_angle']:.1f}°")
for t in result['per_transition']:
    bar = '█' * int(t['angle_degrees'] / 3)
    print(f"  Layers {t['layers']}: {t['angle_degrees']:.1f}° (cos={t['cosine']:.4f}) {bar}")

## Unembedding Direction Trajectory

Track alignment with a specific token's unembedding direction.

In [ ]:
for tok_id in [1, 15, 50]:
    result = unembed_direction_trajectory(model, tokens, position=-1, token_id=tok_id)
    print(f"  Token {tok_id}:")
    for p in result['per_layer']:
        print(f"    Layer {p['layer']}: cos={p['cosine']:.4f}, proj={p['projection']:.4f}")

## Dominant Direction Evolution

Which token does the model predict at each layer?

In [ ]:
result = dominant_direction_evolution(model, tokens, position=-1)
print(f"Prediction changes: {result['n_changes']}")
print(f"Final token: {result['final_token']}")
for p in result['per_layer']:
    flag = ' [CHANGED]' if p['changed'] else ''
    print(f"  Layer {p['layer']}: token={p['top_token']}, "
          f"logit={p['top_logit']:.4f}{flag}")

## Direction Tracking Summary

In [ ]:
result = residual_direction_tracking_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")